In [2]:
import airr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import pandas as pd
import sys
sys.path.append('/mnt/md0/s236922/cowell-lab/repertoire_zoo/')
import repertoire_zoo.bird as br
import repertoire_zoo.giraffe as gr
from scipy import stats # kruskal, chisquare, chi2_contingency, fisher_exact

In [3]:
project_id='c0aef9ef-362d-46fd-abba-39ac0945bd66'
job_id='1a011c26-48a1-4a2d-8dce-47faa65e1b8b-007'

data_dir = '/mnt/md0/Projects/MonsonLab/COVVAX/'
repcalc_dir = data_dir + 'vdjserver/'+job_id+'/'
analysis_dir = data_dir + 'analysis/'
fig_dir = analysis_dir + 'figures/'

data = airr.read_airr(repcalc_dir + 'repertoires.airr.json')
sample_info_df = br.read_repertoire_info_airr(repcalc_dir + 'repertoires.airr.json')
repertoire_group_df = br.read_repertoire_group_info_airr(repcalc_dir + 'repertoires.airr.json')

groups = [
    (repertoire_group_df.loc[i,'repertoire_group_id'], repertoire_group_df.loc[i,'name']) 
    for i in range(len(repertoire_group_df))
]

Total Number of Repertoire Groups:  22


In [ ]:
neutralization_diagnosis_groups = sample_info_df.groupby('diagnosis')['disease_state'].apply(list)[['Healthy Control', 'Heme Cancer', 'Neuroimmunology', 'Solid Cancer']]

# Count lows/highs per group
counts = {group: pd.Series(vals).value_counts() for group, vals in neutralization_diagnosis_groups.to_dict().items()}
table = pd.DataFrame(counts).fillna(0).astype(int).T  # contingency table

print(table, '\n')

# Run chi-squared test
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"Chi2 = {chi2:.3f}, p = {p:.4f}, dof = {dof}")
print("Expected counts:\n", expected)


                 high  low
Healthy Control    11    1
Heme Cancer        18    3
Neuroimmunology    30    4
Solid Cancer       90    8 

Chi2 = 0.960, p = 0.8110, dof = 3
Expected counts:
 [[10.83636364  1.16363636]
 [18.96363636  2.03636364]
 [30.7030303   3.2969697 ]
 [88.4969697   9.5030303 ]]


In [17]:
for i in range(1,len(table)):
    odds_ratio, p_val = stats.fisher_exact(table)
    print(table.iloc[[0,i],:])
    print(f"Fisher’s Exact Test: Odds Ratio={odds_ratio:.3f}, p={p_val:.3f}\n")

                 high  low
Healthy Control    11    1
Heme Cancer        18    3
Fisher’s Exact Test: Odds Ratio=0.017, p=0.764

                 high  low
Healthy Control    11    1
Neuroimmunology    30    4
Fisher’s Exact Test: Odds Ratio=0.017, p=0.765

                 high  low
Healthy Control    11    1
Solid Cancer       90    8
Fisher’s Exact Test: Odds Ratio=0.017, p=0.770



In [19]:
for i in range(len(table)):
    for ii in range(i+1, len(table)):
        odds_ratio, p_val = stats.fisher_exact(table)
        print(table.iloc[[i,ii],:])
        print(f"Fisher’s Exact Test: Odds Ratio={odds_ratio:.3f}, p={p_val:.3f}\n")

                 high  low
Healthy Control    11    1
Heme Cancer        18    3
Fisher’s Exact Test: Odds Ratio=0.017, p=0.768

                 high  low
Healthy Control    11    1
Neuroimmunology    30    4
Fisher’s Exact Test: Odds Ratio=0.017, p=0.767

                 high  low
Healthy Control    11    1
Solid Cancer       90    8
Fisher’s Exact Test: Odds Ratio=0.017, p=0.766

                 high  low
Heme Cancer        18    3
Neuroimmunology    30    4
Fisher’s Exact Test: Odds Ratio=0.017, p=0.767

              high  low
Heme Cancer     18    3
Solid Cancer    90    8
Fisher’s Exact Test: Odds Ratio=0.017, p=0.774

                 high  low
Neuroimmunology    30    4
Solid Cancer       90    8
Fisher’s Exact Test: Odds Ratio=0.017, p=0.761



# Neutralization values

In [6]:
df = pd.read_csv('./Project_COVVAX_Detail_De-Identified_COVVAX_from_PCA.csv', encoding = 'latin')
df = df[['BR Code ', 'PPID', 'To be used as apart of the 166 samples\nRemoved no bulk sort, removed B cell depletion, and removed none/ bad file name ',
         'Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)', 'Low or High neutralization ', 'VNTWHO- Actual Neutralization value',
         'SARS-CoV-2 IgG II']]
df = df[df['To be used as apart of the 166 samples\nRemoved no bulk sort, removed B cell depletion, and removed none/ bad file name '] == 'YES']
df

,BR Code,PPID,"To be used as apart of the 166 samples\nRemoved no bulk sort, removed B cell depletion, and removed none/ bad file name",Ben Greenberg bin designation (Diagnosis),Low or High neutralization,VNTWHO- Actual Neutralization value,SARS-CoV-2 IgG II
1,4492,3193,YES,Solid Cancer,low,16.0000,0.9
3,4505,3196,YES,Solid Cancer,low,16.0000,0.9
4,4518,283,YES,Neuroimmunology,low,16.0000,2.1
5,4538,3203,YES,Heme Cancer,high,543.1074,3579.1
6,4539,3204,YES,Heme Cancer,high,41.0286,136.8
...,...,...,...,...,...,...,...
194,5212,3423,YES,Neuroimmunology,high,153.0000,1462.6
195,5217,3307,YES,Solid Cancer,high,279.5508,2777.5
197,5219,3508,YES,Solid Cancer,high,1700.0000,184216.0
198,5220,3226,YES,Heme Cancer,low,16.0000,92.6


In [4]:
f_stat, p_val = stats.f_oneway(df[df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Healthy Control']['VNTWHO- Actual Neutralization value'],
                               df[df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Solid Cancer']['VNTWHO- Actual Neutralization value'],
                               df[df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Neuroimmunology']['VNTWHO- Actual Neutralization value'],
                               df[df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Heme Cancer']['VNTWHO- Actual Neutralization value'])
print(f"ANOVA: F={f_stat:.3f}, p={p_val:.3f}")

ANOVA: F=0.501, p=0.682


In [5]:
f_stat, p_val = stats.f_oneway(df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Healthy Control') & (df['VNTWHO- Actual Neutralization value']>16)]['VNTWHO- Actual Neutralization value'],
                               df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Solid Cancer') & (df['VNTWHO- Actual Neutralization value']>16)]['VNTWHO- Actual Neutralization value'],
                               df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Neuroimmunology') & (df['VNTWHO- Actual Neutralization value']>16)]['VNTWHO- Actual Neutralization value'],
                               df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Heme Cancer') & (df['VNTWHO- Actual Neutralization value']>16)]['VNTWHO- Actual Neutralization value'])
print(f"ANOVA: F={f_stat:.3f}, p={p_val:.3f}")

ANOVA: F=0.404, p=0.750


In [28]:
df[df['VNTWHO- Actual Neutralization value']>16]

,BR Code,PPID,"To be used as apart of the 166 samples\nRemoved no bulk sort, removed B cell depletion, and removed none/ bad file name",Ben Greenberg bin designation (Diagnosis),Low or High neutralization,VNTWHO- Actual Neutralization value
5,4538,3203,YES,Heme Cancer,high,543.1074
6,4539,3204,YES,Heme Cancer,high,41.0286
9,4569,3219,YES,Heme Cancer,high,31.2930
10,4573,3223,YES,Solid Cancer,high,322.0000
11,4574,3222,YES,Solid Cancer,high,270.5106
...,...,...,...,...,...,...
193,5210,3377,YES,Solid Cancer,high,35.4654
194,5212,3423,YES,Neuroimmunology,high,153.0000
195,5217,3307,YES,Solid Cancer,high,279.5508
197,5219,3508,YES,Solid Cancer,high,1700.0000


In [30]:
df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Healthy Control') & (df['VNTWHO- Actual Neutralization value']>16)]['VNTWHO- Actual Neutralization value']

29       37.5516
31      744.0000
32       82.7526
33       63.9768
58      630.0000
59     1700.0000
68      936.7038
74      395.6826
81      335.1828
84      250.3440
103    1700.0000
Name: VNTWHO- Actual Neutralization value, dtype: float64

## 'SARS-CoV-2 IgG II'

In [ ]:
f_stat, p_val = stats.f_oneway(df[df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Healthy Control']['SARS-CoV-2 IgG II'],
                               df[df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Solid Cancer']['SARS-CoV-2 IgG II'],
                               df[df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Neuroimmunology']['SARS-CoV-2 IgG II'],
                               df[df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Heme Cancer']['SARS-CoV-2 IgG II'])
print(f"ANOVA: F={f_stat:.3f}, p={p_val:.3f}")

ANOVA: F=0.866, p=0.460


In [8]:
f_stat, p_val = stats.f_oneway(df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Healthy Control') & (df['VNTWHO- Actual Neutralization value']>16)]['SARS-CoV-2 IgG II'],
                               df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Solid Cancer') & (df['VNTWHO- Actual Neutralization value']>16)]['SARS-CoV-2 IgG II'],
                               df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Neuroimmunology') & (df['VNTWHO- Actual Neutralization value']>16)]['SARS-CoV-2 IgG II'],
                               df[(df['Ben Greenberg \x93bin\x94 designation\xa0(Diagnosis)']=='Heme Cancer') & (df['VNTWHO- Actual Neutralization value']>16)]['SARS-CoV-2 IgG II'])
print(f"ANOVA: F={f_stat:.3f}, p={p_val:.3f}")

ANOVA: F=1.005, p=0.392
